In [ ]:
import pandas as pd
import numpy as np
import csv

In [ ]:
mov=pd.read_csv(r"/content/output.csv")

In [ ]:
mov.head()

In [ ]:
print(mov.shape)

In [ ]:
mov.info()

In [ ]:
mov.isnull().sum()

In [ ]:
mov.dropna(inplace=True)

In [ ]:
mov.duplicated().sum()

In [ ]:
mov.iloc[1]

In [ ]:
import ast

In [ ]:
mov.head()

In [ ]:
mov.head()

In [ ]:
mov["case_text"]=mov["case_text"].apply(lambda x:x.split())

In [ ]:
mov["case_title"]=mov["case_title"].apply(lambda x:x.split())

In [ ]:
mov["case_text"]=mov["case_text"].apply(lambda x:[i.replace(" ","") for i in x])
mov["case_title"]=mov["case_title"].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
mov.head()

In [ ]:
mov["tags"]=mov["case_title"]+mov["case_text"]
mov.head()

In [ ]:
new_mov=mov[["case_id","case_title","tags"]]
new_mov

In [ ]:
new_mov["tags"]=new_mov["tags"].apply(lambda x:" ".join(x))
new_mov

In [ ]:
new_mov["tags"]=new_mov["tags"].apply(lambda x:x.lower())

In [ ]:
new_mov.head()

In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
new_mov["tags"]=new_mov["tags"].apply(stem)
new_mov

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
cv=TfidfVectorizer(max_features=5000,stop_words="english")

In [ ]:
print(cv.get_stop_words())

In [ ]:
cv.fit_transform(new_mov["tags"]).toarray().shape

In [ ]:
temp=TfidfVectorizer(max_features=5000,stop_words="english")

In [ ]:
temp=temp.fit_transform(new_mov["tags"]).toarray()
temp

In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [ ]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
new_mov["tags"]=new_mov["tags"].apply(stem)
new_mov

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords


In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))
    words = nltk.word_tokenize(text)
    return ' '.join([word for word in words if word.lower() not in stop_words])
new_mov['tags'] = new_mov['tags'].apply(remove_stopwords)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
sim=cosine_similarity(temp)

In [ ]:
sim

In [ ]:
def retrieve(case,n):
    case_id=new_mov[new_mov["case_id"]==case].index[0]
    distances=sim[case_id]
    case_list=sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:n+1]
    case = [new_mov.iloc[score[0]]['case_id'] for score in case_list]
    print(case)

In [ ]:
#to retrieve top 5 cases relevant to case 2
retrieve("Case2",5)

In [ ]:
sim[1]

In [ ]:
retrieve("Case7",10)

In [ ]:
import csv
import os
import re
from collections import defaultdict

In [ ]:
# Load data from CSV file
def load_data(file_path):
    documents = []
    with open(file_path, 'r', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile, delimiter=',')
        for row in reader:
            documents.append(row)
    return documents

In [ ]:
documents = load_data("/content/output.csv")
print(documents)

In [ ]:
def tokenize(text):
    return re.findall(r'\w+', text.lower())

In [ ]:
# Function to index document
def index_documents(documents):
    inverted_index = defaultdict(list)
    doc_ids = []

    for i, doc in enumerate(documents):
        doc_ids.append(i)
        tokens = tokenize(doc['case_title'] + " " + doc['case_text'])
        for token in tokens:
            inverted_index[token].append(i)

    return inverted_index, doc_ids

In [ ]:
inverted_index, doc_ids = index_documents(documents)
print(inverted_index, doc_ids)

In [ ]:
# Function to search by keyword
def keyword_search(query, inverted_index, doc_ids, documents):
    query_tokens = tokenize(query)
    relevant_docs = set(doc_ids)

    for token in query_tokens:
        if token in inverted_index:
            relevant_docs = relevant_docs.intersection(set(inverted_index[token]))

    if relevant_docs:
        print("Search results:")
        for doc_id in relevant_docs:
            print(f"Case ID: {documents[doc_id]['case_id']}")
            print(f"Case Title: {documents[doc_id]['case_title']}")
            print(f"Case Text: {documents[doc_id]['case_text']}")
            print()
    else:
        print("No matching documents found.")

In [ ]:
query = "Australia"
keyword_search(query, inverted_index, doc_ids, documents)

In [ ]:
def search_title(query, documents):
    found = False
    for doc in documents:
        if query == doc['case_title']:
            print("Search results:")
            print(f"Case ID: {doc['case_id']}")
            print(f"Case Title: {doc['case_title']}")
            print(f"Case Text: {doc['case_text']}")
            print()
            found = True
            break

    if not found:
        print("No matching documents found.")

In [ ]:
query = "Barton v Croner Trading Pty Ltd (1984) 3 FCR 95"
search_title(query, documents)

In [ ]:
def wildcard_query_search(inverted_index, documents, query):
    wildcard = query
    wildcard = wildcard.lower()

    # Check if the wildcard is a prefix or suffix
    if wildcard.startswith("*"):
        pattern = re.compile(wildcard.replace("*", ".*") + "$")  # Suffix wildcard
    elif wildcard.endswith("*"):
        pattern = re.compile("^" + wildcard.replace("*", ".*"))  # Prefix wildcard
    else:
        print("Invalid wildcard query format. Please use '*' as prefix or suffix.")
        return

    matching_docs = []

    for token, doc_ids in inverted_index.items():
        if pattern.match(token):
            matching_docs.extend(doc_ids)

    matching_docs = list(set(matching_docs))  # Remove duplicates
    if matching_docs:
        print("Search results:")
        for doc_id in matching_docs:
            print(f"Case ID: {documents[doc_id]['case_id']}")
            print(f"Case Title: {documents[doc_id]['case_title']}")
            print(f"Case Text: {documents[doc_id]['case_text']}")
            print()
    else:
        print("No matching documents found.")

In [ ]:
#suffix wildcard
query = "*wealth"
wildcard_query_search(inverted_index, documents, query)

In [ ]:
#prefix wildcard
query = "heal*"
wildcard_query_search(inverted_index, documents, query)

In [ ]:
def phrase_query_search(documents, query):
    found = False
    for doc in documents:
        text = doc['case_title'] + " " + doc['case_text']
        if re.search(r'\b' + re.escape(query) + r'\b', text):
            print("Search results:")
            print(f"Case ID: {doc['case_id']}")
            print(f"Case Title: {doc['case_title']}")
            print(f"Case Text: {doc['case_text']}")
            print()
            found = True

    if not found:
        print("No matching documents found.")

In [ ]:
query = 'free rider'
phrase_query_search(documents, query)

In [ ]:
def boolean_query_search(query, inverted_index, doc_ids, documents):

    query_tokens = query.split()

    if len(query_tokens) < 3 or len(query_tokens) % 2 == 0:
        print("Invalid query format. Please provide a valid boolean query.")
        return

    relevant_docs = set(doc_ids)
    for i in range(1, len(query_tokens), 2):
        operator = query_tokens[i - 1].upper()
        term = query_tokens[i]
        term_docs = set()

        if term in inverted_index:
            term_docs = set(inverted_index[term])

        if operator == "AND":
            relevant_docs = relevant_docs.intersection(term_docs)
        elif operator == "OR":
            relevant_docs = relevant_docs.union(term_docs)
        elif operator == "NOT":
            relevant_docs = relevant_docs.difference(term_docs)

    # Ensure that only documents containing all query terms are considered
    for doc_id in relevant_docs.copy():
        doc_text = documents[doc_id]['case_title'] + ' ' + documents[doc_id]['case_text']
        if all(term in doc_text for term in query_tokens[::2]):
            print(f"Case ID: {documents[doc_id]['case_id']}")
            print(f"Case Title: {documents[doc_id]['case_title']}")
            print(f"Case Text: {documents[doc_id]['case_text']}")
            print()

    if not relevant_docs:
        print("No matching documents found.")




In [ ]:
boolean_query_search('Health And Ageing', inverted_index, doc_ids, documents,)